# 🎙️ SadTalker — Teacher Tati Avatar
**Anima o rosto da professora sincronizando com o áudio do TTS.**

> ⚠️ Antes de começar: vá em **Runtime → Change runtime type → T4 GPU**

In [ ]:
# ── CÉLULA 1: Verifica GPU ───────────────────────────────────────────
import torch
print('GPU disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Dispositivo:', torch.cuda.get_device_name(0))
else:
    print('SEM GPU! Va em Runtime > Change runtime type > T4 GPU')


In [ ]:
# ── CÉLULA 2: Instala dependências (versões compatíveis) ─────────────
# Ordem importa: numpy primeiro, depois pacotes que dependem dele
!pip install -q 'numpy<2.0'
!apt-get install -qq ffmpeg

# imageio>=2.33 exigido pelo scikit-image 0.25+, ffmpeg junto
!pip install -q 'imageio>=2.33,<2.35' imageio-ffmpeg

# scikit-image sem fixar versão (pega a mais recente compatível)
!pip install -q scikit-image

!pip install -q 'librosa==0.9.2'
!pip install -q face_alignment==1.3.5
!pip install -q basicsr facexlib realesrgan
!pip install -q flask flask-cors pyngrok pydub

import numpy as np, imageio, skimage
print(f'numpy={np.__version__}  imageio={imageio.__version__}  scikit-image={skimage.__version__}')
if int(np.__version__.split('.')[0]) >= 2:
    print('AVISO: NumPy ainda 2.x — clique em Runtime > Restart e rode so esta celula de novo!')
else:
    print('OK: todas as versoes compativeis!')


In [ ]:
# ── CÉLULA 3: Clona SadTalker e baixa modelos ───────────────────────
import os
if not os.path.exists('/content/SadTalker'):
    !git clone https://github.com/OpenTalker/SadTalker.git /content/SadTalker
    print('SadTalker clonado!')
else:
    print('SadTalker ja existe, pulando clone.')
os.chdir('/content/SadTalker')
!mkdir -p checkpoints weights/other gfpgan/weights
print('Baixando checkpoint 256px...')
!wget -q --show-progress -O checkpoints/SadTalker_V0.0.2_256.safetensors 'https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/SadTalker_V0.0.2_256.safetensors'
print('Baixando checkpoint 512px...')
!wget -q --show-progress -O checkpoints/SadTalker_V0.0.2_512.safetensors 'https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/SadTalker_V0.0.2_512.safetensors'
print('Baixando modelos BFM...')
!wget -q -O weights/other/BFM_Fitting.zip 'https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/BFM_Fitting.zip'
!unzip -q -o weights/other/BFM_Fitting.zip -d weights/other/
!wget -q -O weights/other/hub.zip 'https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/hub.zip'
!unzip -q -o weights/other/hub.zip -d weights/other/
print('Baixando GFPGAN...')
!wget -q --show-progress -O gfpgan/weights/GFPGANv1.4.pth 'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth'
print('Todos os modelos baixados!')
!ls -lh checkpoints/


In [ ]:
# ── CÉLULA 4: Patch compatibilidade NumPy 2.x ───────────────────────
import os, glob
patches = [
    ('np.VisibleDeprecationWarning', 'DeprecationWarning'),
    ('np.bool,',   'np.bool_,'),
    ('np.bool)',   'np.bool_)'),
    ('np.int,',    'np.int_,'),
    ('np.int)',    'np.int_)'),
    ('np.float,',  'np.float64,'),
    ('np.float)',  'np.float64)'),
    ('np.complex,','np.complex128,'),
    ('np.object,', 'np.object_,'),
    ('np.str,',    'np.str_,'),
]
py_files = glob.glob('/content/SadTalker/**/*.py', recursive=True)
fixed = []
for fpath in py_files:
    try:
        txt = open(fpath).read()
        new_txt = txt
        for old, new in patches:
            new_txt = new_txt.replace(old, new)
        if new_txt != txt:
            open(fpath, 'w').write(new_txt)
            fixed.append(os.path.relpath(fpath, '/content/SadTalker'))
    except Exception:
        pass
if fixed:
    print(f'Patch aplicado em {len(fixed)} arquivo(s):')
    for f in fixed: print(f'  {f}')
else:
    print('Nenhum patch necessario')


In [ ]:
# ── CÉLULA 5: Upload da foto da Tati ────────────────────────────────
from google.colab import files
import os
os.makedirs('/content/SadTalker/inputs', exist_ok=True)
print('Faca upload da foto da Tati (tati.jpg ou tati.png):')
uploaded = files.upload()
TATI_PHOTO = ''
for fname, data in uploaded.items():
    ext = fname.split('.')[-1].lower()
    dest = f'/content/SadTalker/inputs/tati.{ext}'
    open(dest, 'wb').write(data)
    TATI_PHOTO = dest
    print(f'Foto salva em: {dest}')
os.environ['TATI_PHOTO'] = TATI_PHOTO


In [ ]:
# ── CÉLULA 6: Teste local (opcional) ────────────────────────────────
import os, glob
os.chdir('/content/SadTalker')
TATI_PHOTO = os.environ.get('TATI_PHOTO', '/content/SadTalker/inputs/tati.jpg')
!python inference.py \
    --driven_audio examples/driven_audio/bus_chinese.wav \
    --source_image {TATI_PHOTO} \
    --result_dir /content/test_output \
    --still --preprocess full --enhancer gfpgan --size 256
from IPython.display import Video, display
videos = sorted(glob.glob('/content/test_output/**/*.mp4', recursive=True))
if videos:
    print(f'Video gerado: {videos[-1]}')
    display(Video(videos[-1], embed=True, width=400))
else:
    print('Nenhum video encontrado. Veja os erros acima.')


In [ ]:
# ── CÉLULA 7: SERVIDOR FLASK + NGROK ────────────────────────────────
# Deixe esta celula rodando enquanto usar o Streamlit
# Token GRATUITO em: https://dashboard.ngrok.com/get-started/your-authtoken

NGROK_TOKEN = ''  # <- cole seu token aqui

import os, sys, base64, tempfile, subprocess, threading, time, traceback, glob
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok, conf

SADTALKER_DIR = '/content/SadTalker'
TATI_PHOTO    = os.environ.get('TATI_PHOTO', '/content/SadTalker/inputs/tati.jpg')
OUTPUT_DIR    = '/content/sadtalker_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(SADTALKER_DIR)
sys.path.insert(0, SADTALKER_DIR)

flask_app = Flask(__name__)
CORS(flask_app)

def run_sadtalker(audio_path, output_dir):
    cmd = [
        'python', f'{SADTALKER_DIR}/inference.py',
        '--driven_audio', audio_path,
        '--source_image', TATI_PHOTO,
        '--result_dir',   output_dir,
        '--still', '--preprocess', 'full',
        '--enhancer', 'gfpgan',
        '--size', '256',
        '--expression_scale', '1.0',
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=SADTALKER_DIR)
    if result.returncode != 0:
        raise RuntimeError(result.stderr[-2000:])
    mp4s = sorted(glob.glob(f'{output_dir}/**/*.mp4', recursive=True), key=os.path.getmtime, reverse=True)
    if not mp4s:
        raise RuntimeError('Nenhum video gerado.')
    return mp4s[0]

@flask_app.route('/health')
def health():
    import torch
    gpu = round(torch.cuda.memory_allocated()/1024**2) if torch.cuda.is_available() else 0
    return jsonify({'status': 'ok', 'gpu_mb': gpu})

@flask_app.route('/generate', methods=['POST'])
def generate():
    try:
        data = request.get_json(force=True)
        audio_b64 = data.get('audio_b64', '')
        if not audio_b64:
            return jsonify({'error': 'audio_b64 ausente'}), 400
        audio_bytes = base64.b64decode(audio_b64)
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
            tmp.write(audio_bytes)
            audio_path = tmp.name
        req_dir = tempfile.mkdtemp(dir=OUTPUT_DIR)
        print(f'[SadTalker] Gerando video... audio={len(audio_bytes)/1024:.1f}KB')
        video_path = run_sadtalker(audio_path, req_dir)
        video_b64 = base64.b64encode(open(video_path,'rb').read()).decode()
        os.unlink(audio_path)
        os.unlink(video_path)
        print(f'[SadTalker] Pronto! ({len(video_b64)//1024}KB)')
        return jsonify({'video_b64': video_b64})
    except Exception as e:
        tb = traceback.format_exc()
        print(f'[SadTalker] ERRO:\n{tb}')
        return jsonify({'error': str(e), 'traceback': tb[-1000:]}), 500

ngrok.kill()
if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN
tunnel = ngrok.connect(5000)
public_url = tunnel.public_url
print('=' * 60)
print(f'SERVIDOR ATIVO: {public_url}')
print(f'Copie para o .env do Streamlit:')
print(f'SADTALKER_URL={public_url}')
print('=' * 60)
t = threading.Thread(target=lambda: flask_app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False), daemon=True)
t.start()
time.sleep(2)
print('Servidor rodando. Deixe esta celula ativa!')
import torch
while True:
    gpu = round(torch.cuda.memory_allocated()/1024**2) if torch.cuda.is_available() else 0
    print(f'\rServidor ativo | GPU: {gpu}MB', end='', flush=True)
    time.sleep(10)
